# GÜN 167: C/C++ 'volatile' Anahtar Kelimesi: Ne Zaman Gerekir, Ne Zaman Yetmez? Atomiklik, Torn Reads ve Bellek Bariyerleri

```
ÖZEL LİSANS — TÜM HAKLAR SAKLIDIR
Telif Hakkı (c) 2026 Seydi Eryılmaz (@seydivakkas)
Bu yazılım ve ilgili tüm dosyalar ("Yazılım") yalnızca görüntüleme ve eğitim
amaçlı olarak paylaşılmıştır. Kopyalanamaz, çoğaltılamaz, dağıtılamaz, satılamaz.
```

[![License: All Rights Reserved](https://img.shields.io/badge/license-All%20Rights%20Reserved-red?style=flat-square)](https://github.com/seydivakkas)
![C++20](https://img.shields.io/badge/Standard-C%2B%2B20-blue.svg?style=flat-square)
![ARM Cortex-M](https://img.shields.io/badge/Architecture-ARM%20Cortex--M4%2FM7-orange.svg?style=flat-square)
![Memory Model](https://img.shields.io/badge/Concurrency-volatile%20vs%20atomic-purple.svg?style=flat-square)
![Defense Grade](https://img.shields.io/badge/Domain-Avionics%20%2F%20Critical%20Systems-red.svg?style=flat-square)

---

## 1. Zihinsel Model ve Mimari Büyük Resim (volatile Nedir ve Ne Değildir?)

Gömülü sistem mülakatlarında ve saha arızalarında en çok yanlış anlaşılan C/C++ anahtar kelimesi tartışmasız `volatile`'dır. Birçok geliştirici `volatile`'ı "iş parçacığı güvenliği sağlayan (thread-safe)", "atomik kılan" veya "senkronizasyon çözen" sihirli bir değnek zanneder. Bu yanılgı, savunma sanayii projelerinde rastgele zamanlarda füzelerin yön kaybetmesine veya telemetri paketlerinin bozulmasına yol açan en sinsi böceklerin (Heisenbug) ana kaynağıdır.

### `volatile` Gerçekte Nedir?:
`volatile`, **yalnızca derleyiciye yönelik bir kısıtlamadır (Compiler Directive)**.
Standart C ve C++ dilinde bir değişken `volatile` tanımlandığında derleyiciye şu kesin emir verilir:
> *"Bu bellek hücresi derleyicinin göremediği harici faktörler (donanım çevre birimleri, kesmeler veya doğrudan bellek erişimi - DMA) tarafından her an değiştirilebilir. Bu değişkene yapılan okuma ve yazmaları **ASLA CPU yazmaçlarında (Registers) önbelleğe alma**, **ASLA ölü kod sanıp silme** ve **ASLA sırasını diğer volatile işlemlerle değiştirme**; her seferinde doğrudan fiziksel belleğe git ve `LDR`/`STR` makine komutunu üret!"*

### `volatile` Ne DEĞİLDİR?:
1. **`volatile` Atomik DEĞİLDİR:** Bir `volatile int x;` değişkenine `x++;` dediğinizde bu işlem tek çevrimlik atomik bir işlem haline **GELMEZ**. Hala 3 makine komutudur (`LDR`, `ADD`, `STR`) ve araya giren bir kesme veriyi paramparça eder.
2. **`volatile` Bellek Bariyeri (Memory Barrier) DEĞİLDİR:** İşlemcinin Out-of-Order (sırasız) yürütme motorunu veya donanımsal yazma tamponlarını (Write Buffer) kısıtlamaz; sadece derleyicinin komut sıralamasını yönetir.
3. **`volatile` Çok Çekirdekli (Multicore) Senkronizasyon Sağlamaz:** Donanımsal L1/L2 önbellek tutarlılığı (Cache Coherency) üretmez.

```
   +-------------------------------------------------------------------------+
   |             C/C++ 'volatile' VE DERLEYİCİ MANTIĞI ŞEMASI                 |
   +-------------------------------------------------------------------------+
   
   1. NORMAL DEĞİŞKEN (Non-Volatile) [Derleyici Optimizasyonu -O2 / -O3]:
      RAM'deki Değişken ---> [ İlk Okuma ] ---> CPU Yazmacı (R1)
                                                     |
                                            [ Sürekli R1'i Oku! ] <--- RAM'e ASLA BAKMAZ!
                                            (ISR RAM'i değiştirse dahi göremez -> KİLİTLENİR!)
   
   2. 'volatile' DEĞİŞKEN (Her Erişimde Fiziksel Bellek Zorunlu):
      RAM'deki Değişken <=== [ LDR Komutu ] ===> CPU Çekirdeği (Her Döngüde Gerçek RAM Okunur)
                        <=== [ STR Komutu ] ===> (Donanım ve Kesme Değişiklikleri Anında Görülür)
```

## 2. Derleyici Optimizasyonları (-O2/-O3): Register Caching ve Dead Code Elimination

Modern optimize edici derleyiciler (GCC, Clang, ARMCC) kodu hızlandırmak ve bellek erişim gecikmesini ($10 - 50\ \text{ns}$) sıfırlamak için iki temel varsayımda bulunur:

### 1. Register Caching (Yazmaç Önbelleklemesi):
Aşağıdaki tipik gömülü C kodunu inceleyelim:
```c
bool packet_received = false; // Non-volatile!

void UART_IRQHandler(void) {
    packet_received = true; // Kesme içinde set ediliyor
}

void main_loop(void) {
    while (!packet_received) {
        // Paketin gelmesini bekle
    }
    process_packet();
}
```
Derleyici `-O0` (Hata ayıklama / optimizasyonsuz) modundayken bu kod çalışır. Ancak savunma sanayiinde uçuş yazılımı `-O2` veya `-O3` ile derlendiğinde:
- Derleyici `main_loop` gövdesine bakar: Döngü içinde `packet_received` değerini değiştiren hiçbir satır yoktur.
- Derleyici der ki: *"Belleğe her döngüde `LDR` komutu atmak CPU saatini boşa harcar. Bu değişkeni bir defa CPU yazmacına ($R4$) alayım ve döngüyü `CMP R4, #0` ile yazmaçta döndüreyim."*
- Sonuç: `UART_IRQHandler` kesmesi RAM'deki adresi `true` yapsa bile, ana döngü CPU'nun içindeki $R4$ yazmacına baktığı için değişikliği **ASLA GÖREMEZ**. Sistem sonsuz döngüde kilitlenir!

### 2. Dead Code Elimination (Ölü Kod Temizliği):
```c
uint32_t *status_reg = (uint32_t*) 0x40020010; // Non-volatile MMIO!
*status_reg = 0x01;
*status_reg = 0x02;
```
Derleyici şöyle akıl yürütür: *"Aynı bellek adresine peş peşe iki kez yazılmış; ilk yazılan `0x01` değerini hiç kimse okumadan üzerine `0x02` yazılıyor. O halde `*status_reg = 0x01;` gereksiz bir ölü koddur, onu siliyorum!"*
Oysa o adres bir motor sürücüsü ise, `0x01` motoru açma, `0x02` yön verme komutu olabilir. İlk komut silindiğinde motor arızaya geçer!

## 3. volatile Ne Zaman Zorunludur? (MMIO, ISR Bayrakları, Donanım Durumu)

Gömülü sistem mimarisinde `volatile` kullanımı tam olarak **$3$ senaryoda zorunludur**:

### 1. Bellekle Eşlenmiş Çevresel Yazmaçlar (Memory-Mapped I/O - MMIO):
Donanım yazmaçlarının değeri işlemcinin komutlarından bağımsız olarak elektriksel sinyallerle değişir (örneğin `GPIOA->IDR`, `USART1->SR`):
```c
#define USART1_SR (*((volatile uint32_t*) 0x40011000UL))
// TXE (Transmit data register empty) bayrağını bekle:
while ((USART1_SR & (1 << 7)) == 0); 
```
Burada `volatile` olmazsa derleyici döngüyü sonsuz döngüye çevirir.

### 2. Kesme (ISR) ile Ana Kod (Thread) Arasında Paylaşılan Global Değişkenler:
Bir ISR tarafından güncellenen ve ana döngüde sorgulanan durum bayrakları (Flags):
```c
volatile bool g_missile_abort_command = false;

void EXTI0_IRQHandler(void) {
    EXTI->PR = EXTI_PR_PR0;
    g_missile_abort_command = true; // Kesme bayrağı set eder
}
```

### 3. Çok Görevli (RTOS) veya Çok İşlemcili Sistemlerde Paylaşılan Bellek Alanları:
İki farklı donanım çekirdeğinin (örneğin STM32H7 dual-core Cortex-M7 ve Cortex-M4) paylaştığı ortak SRAM tamponları.

## 4. volatile Ne Zaman Yetmez? (Atomiklik Yoksunluğu, Torn Reads ve Bellek Bariyerleri)

Birçok geliştirici `volatile` eklediğinde kodun eşzamanlılık (concurrency) sorunlarından kurtulduğunu sanır. Bu ölümcül bir hatadır!

### 1. `volatile` İşlemleri Atomik Değildir:
```c
volatile uint32_t packet_counter = 0;

void SysTick_Handler(void) {
    packet_counter++; // KESİNLİKLE GÜVENLİ DEĞİLDİR!
}
```
`packet_counter++` satırı derlendiğinde ARM Assembly çıktısı:
```assembly
LDR  R0, =packet_counter  ; 1. Oku
LDR  R1, [R0]             ; 
ADDS R1, R1, #1           ; 2. Artır
STR  R1, [R0]             ; 3. Yaz
```
Ana görev `LDR` ile `STR` arasındayken bir kesme girerse, sayaç değeri bozulur. `volatile` derleyiciye her seferinde belleğe git der; ancak bu gidiş-gelişin **bölünemez (atomic)** olmasını garanti edemez!

### 2. Yırtık Okuma / Yazma (Torn Read / Torn Write):
32-bit ARM Cortex-M mimarisinde `uint64_t` ($64\ \text{bit}$) bir değişken tek komutta yazılamaz! Donanım iki adet 32-bit `STR` komutu koşturur:
```c
volatile uint64_t g_target_timestamp; // 64-bit zaman damgası
```
Eğer ana görev ilk 32-bit'i yazdıktan hemen sonra bir kesme girer ve `g_target_timestamp` değişkenini okursa, yarısı yeni yarısı eski olan anlamsız bir veri okur (**Torn Read**). Füze koordinatı uzayın bambaşka bir noktası gibi görünür!

### 3. Bellek Yeniden Sıralaması (Memory Reordering) ve Bellek Bariyeri İhtiyacı:
```c
uint8_t telemetry_buffer[256];
volatile bool data_ready = false;

void produce_telemetry(void) {
    memcpy(telemetry_buffer, sensor_data, 256); // 1. Veriyi hazırla
    data_ready = true;                          // 2. Hazır bayrağını yak
}
```
Derleyici `telemetry_buffer`'ın non-volatile olduğunu gördüğünde, optimizasyon amacıyla `data_ready = true;` atamasını `memcpy` tamamlanmadan önceye çekebilir! Tüketici yarım kalmış tamponu okur.
- **Çözüm:** C++20 `std::atomic` veya ARM donanımsal bellek bariyeri `__DMB()` (Data Memory Barrier)!

In [ ]:
# -*- coding: utf-8 -*-
"""
GÜN 167: Kod Bloğu 1 — Derleyici Optimizasyonu ve volatile Simülatörü
(Register Caching vs Bellek Senkronizasyonu)
"""

class CompilerOptimizationSimulator:
    """
    Derleyici Optimizasyon Motoru Simülasyonu:
    - -O0 Modu: Her zaman doğrudan RAM'den okur/yazar.
    - -O3 Modu (Non-volatile): Değişkeni ilk okumada CPU Register'ına önbelleğe alır.
    - -O3 Modu (Volatile): Register caching yasaktır, daima RAM'den LDR/STR üretir.
    """
    def __init__(self):
        self.ram_memory = {"packet_flag": 0}
        self.cpu_register_r4 = None # CPU Önbellek Yazmacı

    def simulate_polling_loop_non_volatile_O3(self, isr_trigger_step: int, max_steps: int = 20) -> Tuple[bool, int]:
        """Non-volatile bayrak: Derleyici R4 yazmacında döner, RAM'deki değişimi GÖREMEZ!"""
        steps = 0
        loop_exited = False

        # Derleyici optimizasyonu: İlk adımda RAM'den R4'e al
        self.cpu_register_r4 = self.ram_memory["packet_flag"]

        for step in range(max_steps):
            steps += 1
            # Simüle edilen Kesme (ISR): Belirli adımda RAM'i günceller
            if step == isr_trigger_step:
                self.ram_memory["packet_flag"] = 1 # RAM güncellendi!

            # CPU Döngüsü: R4 yazmacına bakar (RAM'e BAKMAZ!)
            if self.cpu_register_r4 != 0:
                loop_exited = True
                break

        return loop_exited, steps

    def simulate_polling_loop_volatile_O3(self, isr_trigger_step: int, max_steps: int = 20) -> Tuple[bool, int]:
        """Volatile bayrak: Derleyici R4 önbelleklemesini devre dışı bırakır, her adımda LDR üretir!"""
        steps = 0
        loop_exited = False

        for step in range(max_steps):
            steps += 1
            if step == isr_trigger_step:
                self.ram_memory["packet_flag"] = 1 # RAM güncellendi!

            # VOLATILE EMRİ: Her adımda RAM'den doğrudan oku (LDR komutu)
            fresh_value_from_ram = self.ram_memory["packet_flag"]
            if fresh_value_from_ram != 0:
                loop_exited = True
                break

        return loop_exited, steps

# Deney: 5. Adımda Kesme Geliyor
sim = CompilerOptimizationSimulator()

# 1. Non-Volatile Testi (-O3 Optimizasyonunda Kilitlenme)
exited_non_vol, steps_non_vol = sim.simulate_polling_loop_non_volatile_O3(isr_trigger_step=5, max_steps=20)

print("[*] Derleyici Optimizasyonu ve Register Caching Simülasyonu:")
print(f"  [!] Non-Volatile (-O3): Döngüden Çıkabildi mi? : {exited_non_vol} (Adım: {steps_non_vol})")
print(f"  [!] RAM Değeri: {sim.ram_memory['packet_flag']}, CPU R4 Yazmacı: {sim.cpu_register_r4}")
assert exited_non_vol == False, "Non-volatile değişken register caching nedeniyle kilitlenmelidir!"

# 2. Volatile Testi (Her Adımda Fiziksel LDR Okuması)
sim.ram_memory["packet_flag"] = 0 # Sıfırla
exited_vol, steps_vol = sim.simulate_polling_loop_volatile_O3(isr_trigger_step=5, max_steps=20)

print(f"  [+] Volatile (-O3)    : Döngüden Çıkabildi mi? : {exited_vol} (Adım: {steps_vol})")
assert exited_vol == True, "Volatile değişken kesme güncellemesini anında yakalamalıdır!"
assert steps_vol == 6 # 5. adımda yazıldı, 6. adımda anında çıktı
print("[HÜCRE 6] volatile ve register caching simülatörü başarıyla doğrulandı.")

## 5. Kod Açıklaması: Register Caching ve volatile Çözümü

Yukarıdaki simülasyon `-O3` optimizasyonu altındaki silikon davranışını kanıtlamıştır:
1. `simulate_polling_loop_non_volatile_O3`: Derleyici döngü başında değişkeni CPU yazmacına ($R4$) almış; 5. adımda kesme RAM'deki bayrağı $1$ yapsa dahi CPU yazmaçtaki eski $0$ değerine baktığı için döngüden çıkamamış ve kilitlenmiştir (**Deadlock / Infinite Loop**).
2. `simulate_polling_loop_volatile_O3`: `volatile` emri sayesinde her çevrimde fiziksel `LDR` komutu çalıştırılmış ve 5. adımdaki değişiklik 6. adımda anında yakalanarak döngü sonlandırılmıştır.

In [ ]:
# -*- coding: utf-8 -*-
"""
GÜN 167: Kod Bloğu 2 — 64-bit Değişkenlerde Torn Read (Yırtık Okuma) Simülatörü
(32-bit Cortex-M Veri Yolu Kısıtı)
"""

class TornReadSimulator:
    """
    32-bit ARM Cortex-M Bus Kısıtı:
    - 64-bit (uint64_t) değerler 2 bağımsız 32-bit STR komutuyla yazılır.
    - Düşük 32-bit (Low Word) yazıldıktan hemen sonra araya kesme girerse,
      kesme eski High Word ile yeni Low Word'ün birleşimini okur (TORN READ)!
    """
    def __init__(self):
        self.memory_low32 = 0x00000000
        self.memory_high32 = 0x00000000

    def write_64bit_non_atomic(self, val64: int, interrupt_trigger_callback=None):
        low = val64 & 0xFFFFFFFF
        high = (val64 >> 32) & 0xFFFFFFFF

        # ADIM 1: Düşük 32-bit yazılır
        self.memory_low32 = low

        # [ARAYA KESME GİRİYOR!]
        if interrupt_trigger_callback:
            interrupt_trigger_callback()

        # ADIM 2: Yüksek 32-bit yazılır
        self.memory_high32 = high

    def read_64bit(self) -> int:
        return (self.memory_high32 << 32) | self.memory_low32

# Deney: Füze Zaman Damgası 0x00000000_FFFFFFFF'den 0x00000001_00000000'a geçiyor!
torn_sim = TornReadSimulator()
# Başlangıç değeri: 0x00000000_FFFFFFFF (Düşük: 0xFFFFFFFF, Yüksek: 0x00000000)
torn_sim.write_64bit_non_atomic(0x00000000_FFFFFFFF)

corrupted_read_value = None

def isr_read_probe():
    global corrupted_read_value
    # Kesme tam iki 32-bit yazma işlemi arasında değişkeni okuyor:
    corrupted_read_value = torn_sim.read_64bit()

# Yeni değer yazılıyor: 0x00000001_00000000 (Low=0x00000000, High=0x00000001)
# Yazma anında araya kesme giriyor:
torn_sim.write_64bit_non_atomic(0x00000001_00000000, interrupt_trigger_callback=isr_read_probe)

print()
print("[*] 64-bit Torn Read (Yırtık Okuma) Analizi:")
print(f"  [+] Eski Değer : 0x00000000_FFFFFFFF ({0x00000000_FFFFFFFF})")
print(f"  [+] Yeni Değer : 0x00000001_00000000 ({0x00000001_00000000})")
print(f"  [!] KESMENİN OKUDUĞU YIRTIK DEĞER: 0x{corrupted_read_value:016X} ({corrupted_read_value})")

# Kesme ne eski değeri okudu ne yeni değeri; 0x0 okudu!
assert corrupted_read_value == 0x00000000_00000000, "Torn Read ile verinin 0'a çöktüğü kanıtlandı!"
print("  [!] volatile olmasına rağmen 64-bit veri donanımsal olarak YIRTILDI!")
print("[HÜCRE 8] Torn read simülasyonu başarıyla tamamlandı.")

## 6. Kod Açıklaması: 64-bit Değişkenlerin 32-bit Mimaride Parçalanması

Yukarıdaki deneyde `volatile uint64_t` tanımlanmış bir zaman damgasının nasıl çöktüğü ispatlanmıştır:
1. Değer `0x00000000_FFFFFFFF` seviyesindeyken bir sonraki saniyede `0x00000001_00000000` değerine geçecektir.
2. İşlemci önce düşük 32-bit'e `0x00000000` yazar. Tam o sırada kesme girer ve değişkeni okur.
3. Yüksek 32-bit henüz yazılamadığı için eski `0x00000000` değeriyle birleşir ve kesme **`0x00000000_00000000`** okur!
4. `volatile` kelimesi derleyiciye her iki parçayı da RAM'e yazmasını söylemiştir; ancak 32-bit mimaride bu iki yazmanın arasına girilmesini engelleyememiştir. **Sonuç: `volatile` atomiklik sağlamaz!**

In [ ]:
# -*- coding: utf-8 -*-
"""
GÜN 167: Kod Bloğu 3 — Sabotaj ve Arıza Enjeksiyonu: volatile Sayaç Yarış Durumu (Race Condition)
"""

class VolatileRaceConditionSimulator:
    """
    volatile int counter = 0;
    counter++ işlemi atomik değildir!
    LDR -> ADD -> STR adımları arasına kesme girdiğinde sayaç kayıpları simülasyonu.
    """
    def __init__(self):
        self.shared_counter = 0
        self.lost_updates = 0

    def non_atomic_increment_task(self, isr_intervene: bool = False):
        # 1. Adım: Oku (LDR)
        local_reg = self.shared_counter

        # [SABOTAJ: ARAYA KESME GİRİYOR VE SAYAÇ ARTIRIYOR!]
        if isr_intervene:
            # Kesme kendi içinde counter++ yapıyor
            self.shared_counter += 1

        # 2. Adım: Artır (ADD)
        local_reg += 1

        # 3. Adım: Yaz (STR) - Kesmenin artırdığı değeri ezer!
        self.shared_counter = local_reg

# Simülasyon: 100 Kez Artırma Yapılıyor, 30 Tanesinde Araya Kesme Giriyor
race_sim = VolatileRaceConditionSimulator()

for i in range(100):
    # Her 3 işlemde bir kesme araya girsin
    should_interrupt = (i % 3 == 0)
    race_sim.non_atomic_increment_task(isr_intervene=should_interrupt)

# Beklenen Değer: 100 (Ana Görev) + 34 (Araya giren kesmeler) = 134 olmalıydı
# Ancak araya giren kesmeler ezildiği için değer sadece 100 kalır!
print()
print("[*] volatile++ Yarış Durumu (Race Condition) Sabotaj Testi:")
print(f"  [+] Beklenen Toplam Artış: 134")
print(f"  [!] Gerçekleşen Sayaç     : {race_sim.shared_counter}")
lost = 134 - race_sim.shared_counter
print(f"  [!] Kaybolan / Ezilen Veri : {lost} adet artırma kayboldu!")

assert race_sim.shared_counter == 100, "Kesmelerin yaptığı artırmalar ezilmelidir!"
assert lost == 34, "34 adet kesme artırması kaybolmuştur!"
print("[HÜCRE 10] volatile sayaç yarış durumu (race condition) başarıyla kanıtlandı.")

## 7. Kod Açıklaması: Sabotaj Analizi ve Race Condition Kök Nedenleri

Yukarıdaki sabotaj deneyinde `volatile int counter;` değişkeninin neden tek başına güvenli olmadığı kanıtlanmıştır:
1. Ana görev değişkeni okur (`local_reg = 0`).
2. Araya giren kesme değişkeni artırıp $1$ yapar (`shared_counter = 1`).
3. Ana görev geri döndüğünde elindeki eski $0$ değerini $1$ yapar ve RAM'e yazar (`shared_counter = 1`).
4. **İki artırma yapılmasına rağmen sayaç sadece 1 artmıştır!** Tam 34 adet radar darbe sayımı sessizce çöpe gitmiştir.

In [ ]:
# -*- coding: utf-8 -*-
"""
GÜN 167: Kod Bloğu 4 — C++20 Bellek Modeli: std::atomic, acquire/release ve Donanımsal DMB Bariyeri
"""

class Cpp20AtomicMemoryModel:
    """
    C++20 std::atomic ve Donanımsal Bellek Bariyeri (DMB) Modeli:
    - atomic_store_release: Tampon verisinin belleğe yazılmasını zorlar, sonra bayrağı yakar.
    - atomic_load_acquire  : Bayrağı kontrol eder; bayrak 1 ise tamponun hazır olduğunu garanti eder.
    - atomic_fetch_add     : ARM LDREX/STREX veya kritik bölge ile bölünemez artırma sağlar.
    """
    def __init__(self):
        self.atomic_counter = 0
        self.telemetry_buffer = [0] * 4
        self.data_ready_flag = 0
        self.dmb_barrier_executed = False

    def atomic_fetch_add(self, val: int = 1) -> int:
        """ARM Cortex-M LDREX / STREX Donanımsal Bölünemez Artırma"""
        # Tek çevrimlik atomik işlem: Kesme araya girse dahi donanım çakışmayı yakalar ve tekrarlar
        old = self.atomic_counter
        self.atomic_counter += val
        return old

    def producer_atomic_release(self, payload: List[int]):
        """C++20 memory_order_release Simülasyonu"""
        # 1. Önce veriyi yaz
        for i, b in enumerate(payload):
            self.telemetry_buffer[i] = b

        # 2. Donanımsal Bellek Bariyeri (DMB - Data Memory Barrier)
        # Derleyicinin veya CPU'nun flag yazmasını buffer yazmasının önüne geçirmesini engeller!
        self.dmb_barrier_executed = True

        # 3. Bayrağı atomik set et
        self.data_ready_flag = 1

    def consumer_atomic_acquire(self) -> Optional[List[int]]:
        """C++20 memory_order_acquire Simülasyonu"""
        if self.data_ready_flag == 1:
            # DMB sayesinde buffer'ın tam yazıldığı garantidir
            return list(self.telemetry_buffer)
        return None

# Deney: C++20 Atomik Model Doğrulaması
atomic_model = Cpp20AtomicMemoryModel()

# 1. Atomik Sayaç Testi
for _ in range(134):
    atomic_model.atomic_fetch_add(1)

print()
print("[*] C++20 std::atomic ve Bellek Bariyeri (DMB) Testi:")
print(f"  [+] Atomik Sayaç Değeri: {atomic_model.atomic_counter} (Beklenen: 134)")
assert atomic_model.atomic_counter == 134

# 2. Acquire-Release Veri Transferi
test_payload = [0x55, 0xAA, 0x12, 0x34]
atomic_model.producer_atomic_release(test_payload)
assert atomic_model.dmb_barrier_executed == True

consumed_payload = atomic_model.consumer_atomic_acquire()
print(f"  [+] Acquire İle Okunan Tampon: {[hex(x) for x in consumed_payload]}")
assert consumed_payload == test_payload
print("  [+] Mükemmel: Bellek bariyeri ile sıfır veri yırtılması ve sıfır yeniden sıralama.")
print("[HÜCRE 12] C++20 atomik model ve donanımsal DMB mekanizması başarıyla doğrulandı.")

## 8. Kod Açıklaması: Donanımsal DMB/DSB ve Acquire-Release Senkronizasyonu

Yukarıdaki kod bloğu `volatile`'ın yetmediği yerde C++20 standart çözümünü sunar:
1. **`atomic_fetch_add`:** ARM silikonundaki `LDREX` (Load-Exclusive) ve `STREX` (Store-Exclusive) makine komutlarını taklit ederek sayaç artırma işlemini kesme-güvenli (Interrupt-Safe) hale getirmiştir.
2. **`memory_order_release` & `__DMB()`:** Donanımsal bellek bariyeri koşturarak, 4 baytlık telemetri verisi fiziksel belleğe tam oturmadan `data_ready_flag` bayrağının $1$ olmasını donanım seviyesinde engellemiştir.
3. **`memory_order_acquire`:** Tüketici bayrağı $1$ gördüğü anda arkasındaki tampon verisinin taze ve eksiksiz olduğunu garanti altına almıştır.

In [ ]:
# -*- coding: utf-8 -*-
"""
GÜN 167: Kod Bloğu 5 — ROKETSAN Füze Seyrüsefer ve Güdüm Bilgisayarı Paylaşımlı Bellek Mimarisi
"""

class RocketNavTelemetrySharedMemory:
    """
    ROKETSAN Füze Aviyonik Çekirdeği:
    - 2000 Hz Seyrüsefer Görevi (Thread): 64-bit Yüksek Hassasiyetli Koordinat Üretir
    - 500 Hz Telemetri DMA/ISR (Handler): Koordinatları Telsiz Modülüne Basar
    - Çözüm: Çift Tamponlama (Double Buffering) + Atomik Değişim (Atomic Ping-Pong Pointer)
    """
    def __init__(self):
        # Ping-Pong Tamponları
        self.buffer_ping = {"target_x": 0.0, "target_y": 0.0, "timestamp_us": 0}
        self.buffer_pong = {"target_x": 0.0, "target_y": 0.0, "timestamp_us": 0}
        self.active_write_buffer_idx = 0 # 0: Ping, 1: Pong
        self.atomic_commit_sequence = 0

    def guidance_task_update_coordinates(self, x: float, y: float, timestamp: int):
        """2000 Hz Güdüm Görevi: Pasif Tampona Yazar (Torn Read İmkansızdır!)"""
        target_buf = self.buffer_pong if self.active_write_buffer_idx == 0 else self.buffer_ping
        target_buf["target_x"] = x
        target_buf["target_y"] = y
        target_buf["timestamp_us"] = timestamp

        # Atomik Pointer Değişimi (C++20 atomic_exchange)
        self.active_write_buffer_idx = 1 - self.active_write_buffer_idx
        self.atomic_commit_sequence += 1

    def telemetry_isr_read_latest(self) -> Dict[str, Any]:
        """500 Hz Telemetri ISR: Daima En Son Tamamlanmış Tamponu Okur"""
        # Aktif yazılanın zıttı olan tampon daima tutarlıdır
        read_buf = self.buffer_ping if self.active_write_buffer_idx == 0 else self.buffer_pong
        return dict(read_buf)

# Aviyonik Uçuş Simülasyonu
nav_sys = RocketNavTelemetrySharedMemory()

# 1. Adım: İlk Koordinatlar Yazılıyor (Hedef: X=12500.5m, Y=4500.2m, Zaman=10000us)
nav_sys.guidance_task_update_coordinates(12500.5, 4500.2, 10000)

# 2. Adım: Telemetri ISR anında okuyor
telemetry_packet = nav_sys.telemetry_isr_read_latest()

print()
print("[*] ROKETSAN Seyrüsefer Çift Tamponlu (Double-Buffering) Telemetri Testi:")
print(f"  [+] Okunan Hedef X       : {telemetry_packet['target_x']} m")
print(f"  [+] Okunan Hedef Y       : {telemetry_packet['target_y']} m")
print(f"  [+] Zaman Damgası        : {telemetry_packet['timestamp_us']} us")
print(f"  [+] Atomik Taahhüt Sırası: {nav_sys.atomic_commit_sequence}")

assert telemetry_packet["target_x"] == 12500.5
assert telemetry_packet["target_y"] == 4500.2
assert telemetry_packet["timestamp_us"] == 10000
assert nav_sys.atomic_commit_sequence == 1
print("[HÜCRE 14] Taktik aviyonik seyrüsefer paylaşımlı bellek mimarisi başarıyla doğrulandı.")

## 9. Kod Açıklaması: Aviyonik Güdüm Verisinde Yırtılmasız Transfer

Yukarıdaki taktik aviyonik çözümünde savunma sanayiinin altın standardı uygulanmıştır:
1. Çok elemanlı (Float koordinatlar + 64-bit zaman damgası) veri blokları tek bir `volatile` ile korunamaz.
2. **Ping-Pong (Double-Buffering) Mimarisi:** Güdüm görevi `Pong` tamponuna yazarken, telemetri kesmesi `Ping` tamponundan okur.
3. Yazma işlemi bittiğinde tek bir 32-bit atomik gösterici değişimi (`atomic_exchange`) ile roller değişir. Kesme hiçbir zaman yazılmakta olan tampona dokunamaz; yırtık okuma (**Torn Read**) matematiksel olarak imkansız hale getirilmiştir.

In [ ]:
# -*- coding: utf-8 -*-
"""
GÜN 167: Kod Bloğu 6 — 4 Panelli 300 DPI Görselleştirme Panosu ve Katı Doğrulama Testleri
"""

import os
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np

if 'PNG_FILE' not in globals():
    try:
        PNG_FILE = os.path.join(os.path.dirname(os.path.abspath(__file__)), "gun_167_volatile_ne_zaman_gerekir_ne_zaman_yetmez.png")
    except Exception:
        PNG_FILE = r"c:\Users\seydieryilmaz\Downloads\Olay_Proje\gunluk_mufredat\gun_167\gun_167_volatile_ne_zaman_gerekir_ne_zaman_yetmez.png"

# Matplotlib Karanlık Tema Ayarları
plt.style.use('dark_background')
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['font.size'] = 9.0

fig = plt.figure(figsize=(18, 12), dpi=300, facecolor='#0F172A')
fig.suptitle("GÜN 167: C/C++ 'volatile' Semantiği, Atomiklik, Torn Reads ve Bellek Bariyerleri Panosu",
             fontsize=14, fontweight='bold', color='#38BDF8', y=0.98)

# -----------------------------------------------------------------------------
# PANEL 1: Register Caching vs Volatile Bellek Erişimi (-O3)
# -----------------------------------------------------------------------------
ax1 = fig.add_subplot(2, 2, 1)
ax1.set_facecolor('#1E293B')
ax1.set_title("Panel 1: Register Caching (-O3) vs 'volatile' Bellek Erişimi", fontsize=11, fontweight='bold', color='#F8FAFC', pad=10)
ax1.axis('off')

# Non-volatile Kutusu
ax1.add_patch(patches.FancyBboxPatch((0.05, 0.48), 0.42, 0.45, boxstyle="round,pad=0.02", facecolor='#7F1D1D', edgecolor='#EF4444', lw=2))
ax1.text(0.26, 0.85, "NON-VOLATILE (-O3)", ha='center', va='center', color='#F87171', fontweight='bold', fontsize=10)
ax1.text(0.26, 0.75, "CPU Yazmacına Önbellekleme\n(Register Caching)", ha='center', va='center', color='#FDE68A', fontsize=8.5)
ax1.text(0.26, 0.60, "RAM'e sadece İLK ADIMDA bakar.\nISR RAM'deki bayrağı 1 yapsa dahi,\nCPU R4 yazmacında 0'da döner!\nSONUÇ: SONSUZ KİLİTLENME!", ha='center', va='center', color='#CBD5E1', fontsize=7.5)

# Volatile Kutusu
ax1.add_patch(patches.FancyBboxPatch((0.53, 0.48), 0.42, 0.45, boxstyle="round,pad=0.02", facecolor='#064E3B', edgecolor='#10B981', lw=2))
ax1.text(0.74, 0.85, "VOLATILE (-O3)", ha='center', va='center', color='#34D399', fontweight='bold', fontsize=10)
ax1.text(0.74, 0.75, "Her Adımda LDR Komutu\n(Zorunlu Bellek Okuma)", ha='center', va='center', color='#FDE68A', fontsize=8.5)
ax1.text(0.74, 0.60, "Derleyici önbelleklemeyi iptal eder.\nHer döngüde RAM'den güncel değer okunur.\nKesme değişikliği anında algılanır.\nSONUÇ: BAŞARILI ÇIKIŞ!", ha='center', va='center', color='#CBD5E1', fontsize=7.5)

# Alt Açıklama
ax1.add_patch(patches.Rectangle((0.10, 0.12), 0.80, 0.25, facecolor='#0F172A', edgecolor='#38BDF8', lw=1.5))
ax1.text(0.50, 0.28, "DERLEYİCİ KURALI: 'volatile' SADECE DERLEYİCİ OPTİMİZASYONUNU ENGELLER", ha='center', va='center', color='#38BDF8', fontweight='bold', fontsize=8.5)
ax1.text(0.50, 0.18, "* 'volatile', donanımsal atomiklik veya thread senkronizasyonu SAĞLAMAZ!\n* Sadece derleyicinin değişkeni CPU yazmacında saklamasını yasaklar.", ha='center', va='center', color='#CBD5E1', fontsize=7.5)

ax1.set_xlim(0.0, 1.0)
ax1.set_ylim(0.0, 1.0)

# -----------------------------------------------------------------------------
# PANEL 2: 64-bit Torn Read (Yırtık Okuma) Mekanizması
# -----------------------------------------------------------------------------
ax2 = fig.add_subplot(2, 2, 2)
ax2.set_facecolor('#1E293B')
ax2.set_title("Panel 2: 32-bit Mimaride 64-bit 'Torn Read' (Yırtık Okuma) Fiziği", fontsize=11, fontweight='bold', color='#F8FAFC', pad=10)
ax2.axis('off')

# 64-bit Yazma Aşamaları
ax2.text(0.05, 0.88, "1. ADIM: Düşük 32-bit Yazıldı (STR R0, [Addr+0]):", color='#38BDF8', fontweight='bold', fontsize=8.5)
ax2.add_patch(patches.Rectangle((0.05, 0.74), 0.42, 0.10, facecolor='#0284C7', edgecolor='#FFFFFF', lw=1))
ax2.text(0.26, 0.79, "LOW: 0x00000000 (YENİ)", ha='center', va='center', color='#FFFFFF', fontsize=8, fontweight='bold')
ax2.add_patch(patches.Rectangle((0.50, 0.74), 0.42, 0.10, facecolor='#64748B', edgecolor='#FFFFFF', lw=1))
ax2.text(0.71, 0.79, "HIGH: 0x00000000 (ESKİ)", ha='center', va='center', color='#FFFFFF', fontsize=8)

# Araya Kesme Girişi
ax2.add_patch(patches.Rectangle((0.05, 0.46), 0.87, 0.18, facecolor='#7F1D1D', edgecolor='#EF4444', lw=1.5, linestyle='--'))
ax2.text(0.485, 0.58, "TAM BURADA KESME (ISR) GİRİYOR VE OKUMA YAPIYOR!", ha='center', va='center', color='#FCA5A5', fontweight='bold', fontsize=9)
ax2.text(0.485, 0.50, "Okunan = (ESKİ HIGH << 32) | YENİ LOW = 0x00000000_00000000 (ÇÖP VERİ!)", ha='center', va='center', color='#FDE68A', fontsize=8)

# 2. Adım: Yüksek 32-bit Yazılıyor
ax2.text(0.05, 0.34, "2. ADIM: Yüksek 32-bit Yazıldı (STR R1, [Addr+4]) [ARTIK ÇOK GEÇ!]:", color='#A78BFA', fontweight='bold', fontsize=8.5)
ax2.add_patch(patches.Rectangle((0.05, 0.18), 0.42, 0.10, facecolor='#0284C7', edgecolor='#FFFFFF', lw=1))
ax2.text(0.26, 0.23, "LOW: 0x00000000 (YENİ)", ha='center', va='center', color='#FFFFFF', fontsize=8)
ax2.add_patch(patches.Rectangle((0.50, 0.18), 0.42, 0.10, facecolor='#059669', edgecolor='#FFFFFF', lw=1))
ax2.text(0.71, 0.23, "HIGH: 0x00000001 (YENİ)", ha='center', va='center', color='#FFFFFF', fontsize=8, fontweight='bold')

ax2.set_xlim(0.0, 1.0)
ax2.set_ylim(0.0, 1.0)

# -----------------------------------------------------------------------------
# PANEL 3: volatile++ Yarış Durumu (LDR -> ADD -> STR)
# -----------------------------------------------------------------------------
ax3 = fig.add_subplot(2, 2, 3)
ax3.set_facecolor('#1E293B')
ax3.set_title("Panel 3: volatile++ Neden Atomik Değildir? (Race Condition)", fontsize=11, fontweight='bold', color='#F8FAFC', pad=10)
ax3.axis('off')

steps_data = [
    ("1. Ana Görev: LDR R0, [counter]", "counter = 0 değerini CPU yazmacına (R0=0) okur", "#38BDF8", 0.80),
    ("2. [KESME GİRİYOR!]", "ISR çalışır: counter'ı okur, artırır ve RAM'e 1 yazar!", "#EF4444", 0.60),
    ("3. Ana Görev Devam Eder: ADD R0, #1", "Ana görev kendi R0'ını (0+1=1) artırır", "#F59E0B", 0.40),
    ("4. Ana Görev: STR R0, [counter]", "RAM'e 1 yazar -> KESMENİN YAPTIĞI ARTIŞ EZİLDİ!", "#DC2626", 0.20)
]

for title, desc, col, ypos in steps_data:
    ax3.add_patch(patches.Rectangle((0.05, ypos), 0.90, 0.14, facecolor='#0F172A', edgecolor=col, lw=1.5))
    ax3.text(0.08, ypos + 0.085, title, color=col, fontweight='bold', fontsize=8.5)
    ax3.text(0.08, ypos + 0.035, desc, color='#CBD5E1', fontsize=7.5)

ax3.set_xlim(0.0, 1.0)
ax3.set_ylim(0.05, 1.0)

# -----------------------------------------------------------------------------
# PANEL 4: Çözüm: C++20 std::atomic ve Donanımsal Bellek Bariyeri (DMB)
# -----------------------------------------------------------------------------
ax4 = fig.add_subplot(2, 2, 4)
ax4.set_facecolor('#1E293B')
ax4.set_title("Panel 4: Savunma Çözümü: C++20 Bellek Modeli ve DMB Bariyeri", fontsize=11, fontweight='bold', color='#F8FAFC', pad=10)
ax4.axis('off')

# Üretici Akışı
ax4.add_patch(patches.Rectangle((0.05, 0.55), 0.42, 0.38, facecolor='#0F172A', edgecolor='#10B981', lw=1.5))
ax4.text(0.26, 0.86, "ÜRETİCİ (Görev / Core 1)", ha='center', va='center', color='#34D399', fontweight='bold', fontsize=9)
ax4.text(0.26, 0.76, "1. telemetry_buffer[..] = veri\n2. __DMB() (Data Memory Barrier)\n3. ready.store(1, release)", ha='center', va='center', color='#CBD5E1', fontsize=7.5)
ax4.text(0.26, 0.62, "DMB: Tampon RAM'e yazılmadan\nbayrak ASLA set edilemez!", ha='center', va='center', color='#FDE68A', fontsize=7)

# Tüketici Akışı
ax4.add_patch(patches.Rectangle((0.53, 0.55), 0.42, 0.38, facecolor='#0F172A', edgecolor='#38BDF8', lw=1.5))
ax4.text(0.74, 0.86, "TÜKETİCİ (ISR / Core 2)", ha='center', va='center', color='#38BDF8', fontweight='bold', fontsize=9)
ax4.text(0.74, 0.76, "1. if (ready.load(acquire)) {\n2.   __DMB()\n3.   oku(telemetry_buffer)\n4. }", ha='center', va='center', color='#CBD5E1', fontsize=7.5)
ax4.text(0.74, 0.62, "Acquire: Bayrak 1 ise tamponun\ntam hazır olduğu garantidir!", ha='center', va='center', color='#A7F3D0', fontsize=7)

# Alt Kıyas Tablosu
ax4.add_patch(patches.Rectangle((0.05, 0.10), 0.90, 0.35, facecolor='#1E1B4B', edgecolor='#8B5CF6', lw=1.5))
ax4.text(0.50, 0.38, "NE ZAMAN NE KULLANILMALI? (MİMARİ KARAR TABLOSU)", ha='center', va='center', color='#C084FC', fontweight='bold', fontsize=8.5)
ax4.text(0.50, 0.28, "• Donanım Yazmaçları (MMIO)           -> volatile uint32_t*\n• Kesme ile Paylaşılan Tekil Bayraklar -> volatile bool (veya std::atomic<bool>)\n• Eşzamanlı Sayaç / RMW İşlemleri      -> std::atomic<T> (LDREX / STREX)\n• Çok Elemanlı Veri / Tampon Aktarımı   -> Double-Buffering + DMB / std::atomic", ha='center', va='center', color='#E0E7FF', fontsize=7.5)

ax4.set_xlim(0.0, 1.0)
ax4.set_ylim(0.0, 1.0)

plt.tight_layout()
fig.savefig(PNG_FILE, dpi=300, facecolor='#0F172A', edgecolor='none')
plt.close(fig)

print(f"[Dashboard] 300 DPI Görselleştirme panosu başarıyla oluşturuldu: {PNG_FILE}")

# -----------------------------------------------------------------------------
# KATI DOĞRULAMA TESTLERİ (ASSERTIONS)
# -----------------------------------------------------------------------------
print()
print("[*] GÜN 167 Katı Doğrulama Testleri (Assertions) Başlatılıyor...")

assert os.path.exists(PNG_FILE), "PNG görsel artefaktı bulunamadı!"
png_size = os.path.getsize(PNG_FILE)
print(f"  [+] PNG Varlık ve Boyut Doğrulandı: {png_size:,} bayt.")
assert png_size > 100_000, f"PNG boyutu 100 KB üzerinde olmalı, bulunan: {png_size} bayt"

assert exited_non_vol == False
assert exited_vol == True
assert corrupted_read_value == 0
assert race_sim.shared_counter == 100
assert lost == 34
assert atomic_model.atomic_counter == 134
assert atomic_model.dmb_barrier_executed == True
assert consumed_payload == test_payload
assert telemetry_packet["target_x"] == 12500.5
assert nav_sys.atomic_commit_sequence == 1

print()
print("[PASS] Gün 167: Tüm Katı Doğrulama Testleri Başarıyla Geçti!")

## 10. Görselleştirme Açıklaması ve Mimari Yorumlar

Üretilen 4 panelli yüksek çözünürlüklü (300 DPI) mimari pano, `volatile` ve bellek eşzamanlılığı dinamiklerini ortaya koyar:
1. **Panel 1: Register Caching (-O3) vs 'volatile' Bellek Erişimi:**
   Non-volatile değişkenin CPU yazmacında önbelleğe alınarak sonsuz döngüye düşmesi ile `volatile`'ın her adımda fiziksel `LDR` üreterek kesme güncellemesini anında yakalaması kıyaslanmıştır.
2. **Panel 2: 32-bit Mimaride 64-bit 'Torn Read' (Yırtık Okuma) Fiziği:**
   İki aşamalı 32-bit yazma arasına kesme girdiğinde `0x00000000_00000000` gibi tamamen bozuk bir ara değerin okunması şematize edilmiştir.
3. **Panel 3: volatile++ Neden Atomik Değildir?:**
   `LDR`, `ADD`, `STR` adımları arasına giren kesmelerin 34 adet artırma işlemini nasıl sessizce ezdiği gösterilmiştir.
4. **Panel 4: Savunma Çözümü: C++20 Bellek Modeli ve DMB Bariyeri:**
   `memory_order_release`, `memory_order_acquire` ve `__DMB()` donanım bariyeri ile tampon aktarımının kusursuz senkronizasyonu özetlenmiştir.

## 11. Karşılaştırmalı Analiz: volatile vs const volatile vs std::atomic vs Mutex

| Mekanizma | Derleyici Optimizasyonunu Engeller mi? | Atomiklik Sağlar mı? | Donanımsal Bellek Sıralaması (DMB)? | Çok Çekirdekli Güvenli mi? | Tipik Savunma Kullanımı |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **`volatile`** | **EVET (LDR/STR Zorunlu)** | **HAYIR** | **HAYIR** | **HAYIR** | Donanım Yazmaçları (MMIO), Tekil ISR Bayrakları |
| **`const volatile`** | **EVET** | **HAYIR** | **HAYIR** | **HAYIR** | Salt Okunur Donanım Yazmaçları (`GPIO->IDR`, Çip Seri No) |
| **`std::atomic<T>`** | **EVET** | **EVET (LDREX/STREX)** | **EVET (Acquire/Release)** | **EVET** | Eşzamanlı Sayaçlar, Lock-Free Kuyruklar, Bayraklar |
| **`Mutex / Spinlock`**| **EVET** | **EVET (Kritik Bölge)** | **EVET** | **EVET** | Büyük Veri Yapıları, Dosya Sistemleri, Telemetri Paketleri |

## 12. Donanım ve Çip Seviyesi Ayrıntılar (LDREX/STREX, Exclusive Monitor)

### 1. ARM Exclusive Monitor Donanımı:
ARM Cortex-M3/M4/M7 çekirdeklerinde `std::atomic` operasyonları yazılımsal kilitlerle değil, doğrudan silikon seviyesindeki **Exclusive Monitor** devresiyle koşturulur:
- **`LDREX` (Load Register Exclusive):** Değişkeni bellekten okur ve Exclusive Monitor'e o adres için bir etiket (Tag) koyar.
- Araya bir kesme veya başka bir çekirdek girip o adrese yazarsa, donanım etiketi temizler.
- **`STREX` (Store Register Exclusive):** Yeni değeri yazmaya çalışır. Eğer etiket hala geçerliyse yazma başarılı olur ve $0$ döner; eğer araya kesme girmişse yazma **İPTAL EDİLİR**, $1$ döner ve işlemci döngüde işlemi tekrar dener!
- Bu sayede kesmeler kapatılmadan (`cpsid i` yapılmadan) nanosaniyeler içinde lock-free atomiklik sağlanır.

## 13. Güvenlik, Hata Ayıklama ve Taktik İpuçları (Disassembly İncelemesi)

### 1. Disassembly (Makine Kodu) İle volatile Denetimi:
Bir değişkenin gerçekten `volatile` derlenip derlenmediğini anlamanın tek kesin yolu derlenmiş `.elf` dosyasını `objdump` ile incelemektir:
```assembly
; NON-VOLATILE KOD (-O3):
LDR R0, [R1]      ; Döngüden önce sadece 1 kez okur!
.L2:
CMP R0, #0
BEQ .L2           ; Sonsuz döngü: R0 hiç güncellenmez!

; VOLATILE KOD (-O3):
.L3:
LDR R0, [R1]      ; Her döngüde fiziksel belleğe gider!
CMP R0, #0
BEQ .L3
```

### 2. volatile Gösterici (Pointer) Sözdizimi Tuzağı:
```c
volatile uint32_t * ptr1; // İşaret edilen VERİ volatile (En yaygın MMIO kullanımı)
uint32_t * volatile ptr2; // İşaretçinin KENDİSİ (adresi) volatile
volatile uint32_t * volatile ptr3; // Hem adres hem veri volatile!
```
Donanım yazmaçlarını tanımlarken daima `volatile uint32_t * const` (adresi değişmez, verisi volatile) kullanılmalıdır.

## 14. Günün Mini Uygulaması: volatile, Torn Read ve C++20 Atomik Modeli

Günün mini uygulamasında:
1. `-O3` optimizasyonu altında `volatile` olmayan bir bayrağın CPU yazmacında önbelleğe alınarak nasıl sonsuz döngü ürettiği, `volatile` ile her adımda `LDR` üretilerek sistemin nasıl kurtarıldığı simüle edilmiştir.
2. 32-bit Cortex-M mimarisinde $64\ \text{bit}$ zaman damgası yazılırken araya kesme girmesiyle oluşan **Torn Read (Yırtık Okuma)** arızası modellenmiştir.
3. `volatile++` işleminin atomik olmadığı, araya giren kesmeler nedeniyle 34 adet artırmanın sessizce kaybolduğu doğrulanmıştır.
4. C++20 `std::atomic`, `memory_order_release/acquire` ve donanımsal `__DMB()` bellek bariyeri uygulanmıştır.
5. ROKETSAN füze seyrüsefer bilgisayarı için çift tamponlu (Double-Buffering Ping-Pong) aviyonik telemetri mimarisi test edilmiştir.

## 15. Sık Yapılan Hatalar ve Anti-Pattern'ler

1. **`volatile`'ı Mutex veya Kilit Sanmak:** Çok iş parçacıklı veya kesmeli sistemlerde sayaçları `volatile int` yapıp veri yarışlarına yol açmak.
2. **`volatile` Pointer Tanımını Karıştırmak:** `volatile int*` yerine `int* volatile` yazıp veriyi optimize edilebilir bırakmak.
3. **Bellek Bariyeri Koymadan Tampon Paylaşmak:** Derleyicinin veya CPU'nun `data_ready = true;` bayrağını tampon yazılmadan önceye çekmesine izin vermek.
4. **Her Değişkene Rastgele `volatile` Eklemek:** Gereksiz yere `volatile` kullanarak derleyicinin tüm CPU yazmaç optimizasyonlarını felç etmek ve kod performansını 5 kat düşürmek.

## 16. Mülakat ve Saha Deneyimi Soruları (ASELSAN / ROKETSAN / TUSAŞ)

### Soru 1: C dilinde bir değişkeni 'volatile' yapmak o değişkene yapılan okuma ve yazmaların atomik olduğunu garanti eder mi? Neden?
**Cevap:** Kesinlikle hayır. `volatile`, yalnızca derleyiciye o değişkenin değerini CPU yazmaçlarında önbelleğe almaması ve her erişimde fiziksel belleğe `LDR`/`STR` komutu üretmesi gerektiğini söyler. Ancak `x++` gibi işlemler işlemci seviyesinde 3 adımdır (Oku, Değiştir, Yaz). Eğer okuma ile yazma arasına bir kesme girerse yarış durumu (Race Condition) oluşur. Ayrıca 32-bit bir işlemcide 64-bitlik bir `volatile uint64_t` değişkeni iki bağımsız 32-bit komutla yazıldığından, araya giren kesme yarım yazılmış veriyi okuyarak 'Torn Read' felaketine yol açar. Atomiklik için `volatile` değil; C++20 `std::atomic`, ARM `LDREX`/`STREX` komutları veya kesme maskeleme (`__disable_irq`) kullanılmalıdır.

### Soru 2: Gömülü sistemlerde bir durum bayrağı (Flag) 'volatile' olarak tanımlandığı halde, ana döngü neden bayrağın arkasındaki veri tamponunu yarım veya bozuk okuyabilir?
**Cevap:** Bunun sebebi 'Bellek Sıralaması' (Memory Ordering) ve bellek bariyerlerinin (Memory Barriers) eksikliğidir. `volatile` anahtar kelimesi, iki `volatile` işlem arasındaki sırayı korur; ancak `volatile` olan bayrak ile `non-volatile` olan veri tamponu arasındaki sırayı korumayı garanti etmez. Derleyici veya sırasız çalışan modern bir CPU mimarisi, optimizasyon amacıyla bayrağı set etme komutunu (`flag = 1`), tamponu belleğe kopyalayan döngünün (`memcpy`) önüne alabilir. Bu durumda tüketici bayrağı 1 görür ve tampon henüz RAM'e tam oturmadan okuyarak bozuk veri çeker. Çözüm, bayrağı set etmeden hemen önce bir Donanımsal Veri Bellek Bariyeri (`__DMB()`) koymak veya C++20 `memory_order_release` semantiği kullanmaktır.

### Soru 3: 'const volatile uint32_t *' tanımı C dilinde mantıklı mıdır? Mikrodenetleyicilerde nerede kullanılır?
**Cevap:** Evet, son derece mantıklıdır ve gömülü sistemlerin en temel idiomlarından biridir. `const`, yazılımcının (yazılım kodunun) o bellek adresine veri yazamayacağını (derleme zamanında salt okunur olduğunu) belirtir. `volatile` ise o bellek adresinin değerinin dış dünya (donanım) tarafından her an değiştirilebileceğini belirtir. Bu yapı **Salt Okunur Donanım Yazmaçlarında** kullanılır. En tipik örnekler: `GPIO->IDR` (Giriş Veri Yazmacı - yazılımla pini değiştiremezsiniz ama dışarıdan buton basılınca değeri değişir) veya mikrokontrolcünün fabrikasyon çip benzersiz seri numarası (UID) adresleridir.

## 17. Gün Sonu Özeti ve 3-Cümle Çıkarımı

1. **volatile Nedir?:** Derleyicinin CPU yazmaç önbelleklemesini (Register Caching) ve ölü kod temizliğini engelleyerek her erişimde fiziksel `LDR`/`STR` üretmesini sağlayan kısıtlamadır.
2. **Hangi Problemi Çözer?:** Donanım yazmaçlarının (MMIO) ve kesmelerin (ISR) güncellediği bayrakların `-O2/-O3` optimizasyonlarında sonsuz döngüye kilitlenmesini önler.
3. **Hangi Durumda Yetmez?:** Atomiklik sağlamaz (sayaç yarışları ve 64-bit Torn Read oluşur); tampon aktarımlarında bellek sıralamasını garanti etmek için mutlaka C++20 `std::atomic` ve donanımsal `DMB` bariyerleri gerekir.